# 🎭 Ролевая модель на базе Qwen2.5 (1,5 млрд параметров)

**Нужна видеокарта:** меню «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**.

1. **▶ Дообучить** — ~1,5–2 часа. Прогресс сохраняется на Google Диск (`ai_qwen`). Colab отключился → открой ноутбук снова и нажми ▶ — продолжит с места остановки.
2. **💬 Чат** — когда обучение закончится (или в любой момент — возьмёт последнее сохранение). Персонажа задавай в «🎭 Персонаж и настройки» под окном чата.

Модель «base» — без встроенных отказов. Защита от сексуальных сцен с несовершеннолетними включена всегда.

In [ ]:
#@title ▶ Дообучить Qwen на ролевых данных (после отключения — просто ▶ снова)
import os, sys, torch
SAVE = '/content/drive/MyDrive/ai_qwen'      # здесь хранится прогресс и готовый адаптер
REPO, BRANCH = '/content/ai', 'ccr-d2dd0252-6ksj1l'
if not torch.cuda.is_available():
    print('⚠️ Нет видеокарты! Меню «Среда выполнения» → «Сменить среду выполнения» → T4 GPU → Сохранить. Потом снова ▶')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(SAVE, exist_ok=True)
    if os.path.exists(REPO):
        !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} reset -q --hard FETCH_HEAD
    else:
        !git clone -q -b {BRANCH} https://github.com/r0561952-beep/ai.git {REPO}
    os.chdir(REPO)
    !pip install -q "transformers==5.24.1" "peft==0.22.0" "accelerate==1.16.0" zstandard
    # ролевые данные готовим один раз и храним на Диске
    os.makedirs(f'{SAVE}/data', exist_ok=True)
    if not os.path.exists(f'{SAVE}/data/rp18_ru.jsonl.gz'):
        print('Первый запуск: скачиваю и фильтрую ролевые данные (~5 минут)...')
        !python prepare_rp.py
        !cp data/rp_ru.jsonl.gz data/rp18_ru.jsonl.gz {SAVE}/data/
    else:
        !cp {SAVE}/data/rp_ru.jsonl.gz {SAVE}/data/rp18_ru.jsonl.gz data/
    !python qwen_train.py --out {SAVE} 2>&1 | tee -a {SAVE}/train.log

In [ ]:
#@title 💬 Чат с Qwen (как в ChatGPT)
import os, sys, torch
SAVE = '/content/drive/MyDrive/ai_qwen'      # здесь хранится прогресс и готовый адаптер
REPO, BRANCH = '/content/ai', 'ccr-d2dd0252-6ksj1l'
if not torch.cuda.is_available():
    print('⚠️ Нет видеокарты! Меню «Среда выполнения» → «Сменить среду выполнения» → T4 GPU → Сохранить. Потом снова ▶')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(SAVE, exist_ok=True)
    if os.path.exists(REPO):
        !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} reset -q --hard FETCH_HEAD
    else:
        !git clone -q -b {BRANCH} https://github.com/r0561952-beep/ai.git {REPO}
    os.chdir(REPO)
    !pip install -q "transformers==5.24.1" "peft==0.22.0" "accelerate==1.16.0" "gradio>=6,<7"
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    for m in ['model', 'tokenizer_utils', 'chat', 'web_chat', 'qwen_chat', 'rp_card', 'safety']:
        sys.modules.pop(m, None)
    from web_chat import launch
    adapter = f'{SAVE}/adapter' if os.path.exists(f'{SAVE}/adapter') else None
    print('Модель: Qwen2.5-1.5B' + (' + наш ролевой адаптер' if adapter else ' БЕЗ дообучения (сначала запусти ячейку выше)'))
    print('👇 Открой ссылку «public URL» ниже')
    launch(share=True, qwen_adapter=adapter)

---

In [ ]:
#@title 📈 Журнал обучения
import os
SAVE = '/content/drive/MyDrive/ai_qwen'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
log = f'{SAVE}/train.log'
if os.path.exists(log):
    lines = [l for l in open(log, encoding='utf-8') if l.startswith(('шаг', 'продолжаю', '✅', 'лимит', 'примеров'))]
    print(''.join(lines[-12:]))
else:
    print('Журнала пока нет — сначала запусти обучение.')